# OCSD: same sketch, different backgrounds (for the slides)

One sketch (2 orange buses + 1 wooden bench), OCSD on **Stable Diffusion 1.5 with all five modules M1–M5** (M3 on), exact thesis code and settings (paper_v2 tuning).
M1, M2 objects and M3 identity tokens are made once; then the scene (M4 + M5) is generated with the scene's own background and 3 other backgrounds, same seed, only the background words changed.

**How to run:** Runtime → Change runtime type → any GPU (T4, L4 or A100) → **Run all**. About 15 minutes. A ZIP downloads at the end; it is also on Drive at `KLTN_OCSD/trace_ocsd/`.


In [ ]:
#@title 1. Settings
SID    = "qd_3_medium_003"   #@param {type:"string"}
SEED   = 0                   #@param {type:"integer"}
BGS    = ["on a city street", "on a snowy field", "on a sandy beach"]
BRANCH = "claude/project-thread-srtlwl"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"  #@param {type:"string"}


In [ ]:
#@title 2. Drive, GPU check, code, libraries
import os, subprocess, sys, shutil
from google.colab import drive
drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(gpu.stdout)
if gpu.returncode or int(gpu.stdout.strip().split(",")[-1]) < 14000:
    raise RuntimeError("This needs a GPU with at least 14 GB (T4, L4 or A100): Runtime > Change runtime type > GPU, then Run all")
# OCSD on SD 1.5 (with M3 training) fits in about 12 GB, so a T4, L4 or A100 all work
CODE = "/content/KLTN_OCSD_bgswap"
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/bobbibao/ocsd-thesis.git", CODE], check=True)
print("Code:", subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
os.environ["HF_HOME"] = "/content/hf_cache"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(CODE, "requirements-colab.txt")], capture_output=True, text=True)
print(r.stdout[-1000:], r.stderr[-2000:])
if r.returncode:
    raise RuntimeError("pip install failed (see the text above)")
from ocsd import hfcache
hfcache.repair(os.environ["HF_HOME"])

In [ ]:
#@title 3. Generate: own background + 3 others (about 10 minutes after the models are downloaded)
from ocsd import bg_swap
META = bg_swap.run(DRIVE_ROOT, sid=SID, seed=SEED, bgs=BGS)
for r in META["runs"]: print(r["bg"], "|", r["global_prompt"], "|", r["seconds"], "s")


In [ ]:
#@title 4. Show the result and download the ZIP
from IPython.display import Image, display
out = os.path.join(DRIVE_ROOT, "trace_ocsd", f"bgswap_{SID}_s{SEED}")
display(Image(os.path.join(out, "SHEET.jpg"), width=1400))
from google.colab import files
try:
    files.download(META["zip"])
except Exception as exc:
    print("Get it from Drive:", META["zip"], exc)
print("Done. Disconnect the GPU runtime.")
